In [1]:
import sqlite3
import json
import pandas as pd

# Path to your database
DB_PATH = 'db/results.sqlite'

def verify_database_integrity():
    print(f"Checking database: {DB_PATH}...\n")
    
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.row_factory = sqlite3.Row
        
        # 1. Fetch the most recent result
        cursor = conn.execute("SELECT * FROM Results ORDER BY result_id DESC LIMIT 1")
        row = cursor.fetchone()
        
        if row is None:
            print("❌ Database is empty! Run an experiment first.")
            return

        print(f"✅ Connection successful. Inspecting Result ID: {row['result_id']}\n")

        # --- CHECK 1: OLD DATA (Scalars & Text) ---
        print("--- [1/3] Verifying Legacy Data ---")
        old_keys = ['uq_mech_score', 'uq_avg_entropy', 'full_trace_text', 'predicted_answer']
        
        all_old_ok = True
        for key in old_keys:
            val = row[key]
            if val is not None and val != "":
                print(f"  ✅ {key}: {str(val)[:50]}...")
            else:
                print(f"  ❌ {key} is MISSING or EMPTY!")
                all_old_ok = False
        
        # --- CHECK 2: NEW DATA (Traces & Counts) ---
        print("\n--- [2/3] Verifying New Token-Level Data ---")
        new_keys = ['uq_mech_trace', 'uq_entropy_trace', 'uq_logit_gap_trace', 'gen_len']
        
        all_new_ok = True
        for key in new_keys:
            val = row[key]
            
            # Special check for 'gen_len' (should be int)
            if key == 'gen_len':
                if isinstance(val, int) and val > 0:
                     print(f"  ✅ {key}: {val} (Integer)")
                else:
                    print(f"  ❌ {key}: Invalid or missing integer ({val})")
                    all_new_ok = False
                continue
            
            # Check JSON lists
            try:
                parsed = json.loads(val)
                if isinstance(parsed, list) and len(parsed) > 0:
                    print(f"  ✅ {key}: Found list with {len(parsed)} items. First item: {parsed[0]:.4f}")
                    
                    # Consistency Check
                    if len(parsed) != row['gen_len']:
                        print(f"     ⚠️ WARNING: Trace length ({len(parsed)}) != gen_len ({row['gen_len']})")
                else:
                    print(f"  ❌ {key}: JSON parsed but empty or not a list.")
                    all_new_ok = False
            except TypeError:
                 print(f"  ❌ {key}: Value is None/Null. Did you delete the old DB file?")
                 all_new_ok = False
            except json.JSONDecodeError:
                print(f"  ❌ {key}: Malformed JSON.")
                all_new_ok = False

        # --- CHECK 3: COLUMN EXISTENCE ---
        # Double check the schema actually has the columns
        print("\n--- [3/3] Schema Validation ---")
        columns = [description[0] for description in cursor.description]
        missing_cols = [k for k in new_keys if k not in columns]
        
        if not missing_cols:
            print("  ✅ All new columns exist in table schema.")
        else:
            print(f"  ❌ Missing columns in DB schema: {missing_cols}")

        print("\n" + "="*40)
        if all_old_ok and all_new_ok and not missing_cols:
            print("🎉 SUCCESS: Database schema and data logging are correct!")
        else:
            print("⚠️ ISSUES DETECTED: See errors above.")
            
    except sqlite3.OperationalError as e:
        print(f"❌ SQL Error: {e}")
        print("   (Did you delete the old .sqlite file before running the new code?)")
    finally:
        if 'conn' in locals(): conn.close()

# Run the check
verify_database_integrity()

Checking database: db/results.sqlite...

✅ Connection successful. Inspecting Result ID: 20

--- [1/3] Verifying Legacy Data ---
  ✅ uq_mech_score: 0.9715252311862245...
  ✅ uq_avg_entropy: 0.06333890420441725...
  ✅ full_trace_text:  To determine how many pages James writes in a yea...
  ✅ predicted_answer: 312...

--- [2/3] Verifying New Token-Level Data ---


IndexError: No item with that key